# 03 Canny From Mask Sweep

This notebook verifies that edge maps are derived from instance masks, never RGB. It sweeps line width parameters and saves overlays for B3 acceptance.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from blast_pile_diffusion.data.sample_bundle import SampleBundle
from blast_pile_diffusion.preprocessing.canny_from_mask import mask_to_canny

PREPROCESSED_DIR = Path('../data/preprocessed')
OUTPUT_DIR = Path('../notebooks/outputs/canny_sweep')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bundle_dirs = sorted(p for p in PREPROCESSED_DIR.iterdir() if (p / 'rgb.png').exists())
assert bundle_dirs, f'No bundles found under {PREPROCESSED_DIR}. Run scripts/01_preprocess_unity.py first.'
bundle = SampleBundle.load(bundle_dirs[0])
bundle.sample_key, bundle.rgb.shape, int(bundle.num_instances)


In [ ]:
def overlay_edges(rgb, edges):
    out = rgb.copy()
    edge_mask = edges > 0
    out[edge_mask] = np.array([255, 40, 40], dtype=np.uint8)
    return out

dilate_values = [1, 2, 3, 5]
fig, axes = plt.subplots(1, len(dilate_values), figsize=(4 * len(dilate_values), 4))
for ax, dilate in zip(axes, dilate_values):
    edges = mask_to_canny(bundle.mask, dilate_kernel=dilate)
    ax.imshow(overlay_edges(bundle.rgb, edges))
    ax.set_title(f'dilate={dilate}, edge_px={int((edges > 0).sum())}')
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{bundle.sample_key}_canny_sweep.png', dpi=160)
plt.show()


In [ ]:
edges = mask_to_canny(bundle.mask, dilate_kernel=2)
interior = np.zeros_like(edges, dtype=bool)
for instance_id in np.unique(bundle.mask):
    if instance_id == 0:
        continue
    instance = bundle.mask == instance_id
    interior |= instance & ~(edges > 0)
print({'instances': int(bundle.num_instances), 'edge_pixels': int((edges > 0).sum()), 'interior_pixels': int(interior.sum())})
